# 05 — ML de clasificación: ¿llegó la onda P?
El detector de onda P es la primera pieza de una alerta temprana: clasifica ventanas de 4 s de STEAD en "llegada de la onda P" o "no".
1. **Monitoreo:** ¿sigue sirviendo el modelo **champion**? Se miran la degradación de su AUC en el lote nuevo y el drift (PSI) de sus variables.
2. **Decisión de recalibrado:** si no hay evidencia, el notebook termina y se conserva el champion.
3. **Entrenamiento:** se entrenan 4 algoritmos de MLlib sobre los mismos datos, se comparan en un *holdout* (30 % de las trazas) y el mejor es el **challenger**.
4. **Promoción:** el challenger reemplaza al champion solo si lo supera por un margen mínimo. El modelo se guarda en el Volume `modelos/`.
5. **MLflow:** la comparación queda registrada como experimento.

Como referencia se mide primero el picador clásico STA/LTA sobre las mismas trazas.

## 1. Setup — imports

In [ ]:
# Librerías que usa el notebook.
import json
from datetime import datetime

import mlflow
import numpy as np
import pandas as pd

from pyspark.sql.functions import col, lit, when
from pyspark.sql import functions as F

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier,
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

## 2. Catálogo, parámetros y umbrales

In [ ]:
# Catálogo y esquema; forzar_recalibrado llega como parámetro del job.
catalogo = "grupo06proyectofinal"
esquema = "sismos"
volumen = f"/Volumes/{catalogo}/{esquema}"

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"USE SCHEMA {esquema}")
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

dbutils.widgets.text("forzar_recalibrado", "false")
forzar = dbutils.widgets.get("forzar_recalibrado").lower() == "true"

In [ ]:
# Caso, target y umbrales de monitoreo y promoción.
caso = "clasificacion"
target = "etiqueta"
SEED = 700

UMBRALES = {
    "psi_feature": 0.20,
    "n_features_drift": 2,
    "caida_auc": 0.05,
    "mejora_minima_auc": 0.01,
}

## 3. Función de drift: PSI
Population Stability Index entre la distribución de referencia (datos del champion) y la del lote nuevo. PSI > 0.20 indica un cambio importante.

In [ ]:
# PSI de cada variable entre la referencia y el lote actual.
def psi(referencia, actual, bins=10):
    ref = pd.Series(referencia).dropna().astype(float)
    act = pd.Series(actual).dropna().astype(float)
    cortes = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(cortes) < 3:
        return 0.0
    cortes[0], cortes[-1] = -np.inf, np.inf
    p = np.histogram(ref, cortes)[0] / len(ref)
    q = np.histogram(act, cortes)[0] / len(act)
    p, q = np.clip(p, 1e-4, None), np.clip(q, 1e-4, None)
    return float(np.sum((q - p) * np.log(q / p)))

## 4. Línea base: picador clásico STA/LTA

In [ ]:
# Métricas del picador STA/LTA como línea base.
picks = spark.table("gold_onda_p_picks").withColumn("abs_e", F.abs("error_s"))
fila_sta = picks.agg((100 * F.avg(F.col("error_s").isNotNull().cast("double"))).alias("pct_trazas_detectadas"),
                     F.median("abs_e").alias("error_abs_mediano_s"),
                     (100 * F.sum((F.col("abs_e") < 0.5).cast("double")) / F.greatest(F.count("error_s"), F.lit(1))).alias("pct_error_menor_0_5s")).first()
sta_lta = fila_sta.asDict()
print({k: round(float(v), 3) for k, v in sta_lta.items()})
filas_sta = [("picador_sta_lta", k, float(v), json.dumps({"sta_s": 0.5, "lta_s": 5.0, "umbral": 4.0}),
              datetime.now().isoformat(timespec="seconds"), "05_ml_clasificacion") for k, v in sta_lta.items()]
(spark.createDataFrame(filas_sta, "modelo string, metrica string, valor double, parametros string, fecha string, notebook string")
 .write.mode("overwrite").option("replaceWhere", "notebook = '05_ml_clasificacion'").saveAsTable("ml_metricas"))

## 5. Datos y variables seleccionadas

In [ ]:
# Lee los datos y las variables seleccionadas.
df = spark.table("fe_clasificacion")

seleccion = spark.table("fe_columnas_seleccionadas").filter(col("caso") == caso)
cols_nums = [r["columna"] for r in seleccion.filter(col("tipo") == "num").collect()]

lotes = sorted(r[0] for r in df.select("lote").distinct().collect())
lote_actual = lotes[-1]

print("Lotes:", lotes, "| lote más reciente:", lote_actual)
print("Variables seleccionadas:", cols_nums)

## 6. Champion actual (registro de modelos)

In [ ]:
# Carga el champion vigente y sus metadatos.
registro = spark.table("ml_registro_modelos").filter(col("caso") == caso)
fila_champion = registro.filter(col("promovido")).orderBy(col("version").desc()).first()

if fila_champion is None:
    champion, meta = None, None
else:
    champion = PipelineModel.load(fila_champion["ruta"])
    meta = {"version": fila_champion["version"],
            "cols_num": json.loads(fila_champion["columnas_num"]),
            "cols_cat": json.loads(fila_champion["columnas_cat"]),
            "snapshots": json.loads(fila_champion["snapshots"]),
            "metricas": json.loads(fila_champion["metricas"]),
            "detalle": json.loads(fila_champion["detalle"])}

print("Champion actual:", "v" + str(meta["version"]) if meta else "no existe")

## 7. Monitoreo: degradación de la métrica y drift

In [ ]:
# Mide la degradación del champion y el drift de cada variable.
n_drift, metrica_base, metrica_nueva, degradacion = 0, None, None, False
evaluador_auc = BinaryClassificationEvaluator(labelCol=target, metricName="areaUnderROC")

if champion is not None and lote_actual not in meta["snapshots"]:
    ref = df.filter(col("lote").isin(meta["snapshots"])).select(meta["cols_num"]).toPandas()
    act = df.filter(col("lote") == lote_actual).select(meta["cols_num"]).toPandas()
    psis = {c: psi(ref[c], act[c]) for c in meta["cols_num"]}
    n_drift = sum(v > UMBRALES["psi_feature"] for v in psis.values())
    print("PSI (top 5):", [(c, round(v, 3)) for c, v in sorted(psis.items(), key=lambda kv: -kv[1])[:5]])

    metrica_base = meta["metricas"]["AUC"]
    metrica_nueva = evaluador_auc.evaluate(champion.transform(df.filter(col("lote") == lote_actual)))
    degradacion = metrica_base - metrica_nueva > UMBRALES["caida_auc"]
    print(f"AUC base: {metrica_base:.3f} | AUC en {lote_actual}: {metrica_nueva:.3f} | degradacion: {degradacion}")
elif champion is not None:
    print(f"El champion ya se entrenó con {lote_actual}: no hay datos nuevos que monitorear")

## 8. Decisión de recalibrado
Se recalibra si no hay champion, si la métrica se degradó, si hay drift en 2 o más variables o si el job lo fuerza. La decisión queda en `ml_monitoreo`.

In [ ]:
# Decide si recalibrar: sin champion, degradación, drift o pedido del job.
if champion is None:
    reentrenar, motivo = True, "No existe champion: primer entrenamiento"
else:
    drift = n_drift >= UMBRALES["n_features_drift"]
    reentrenar = bool(degradacion or drift or forzar)
    senales = [("degradacion de la metrica", degradacion), (f"drift en {n_drift} variables", drift), ("forzado por parametro", forzar)]
    motivos = [texto for texto, activa in senales if activa]
    motivo = " + ".join(motivos) if motivos else "Sin cambios relevantes: se conserva el champion"

print("DECISION:", "RECALIBRAR" if reentrenar else "SIN CAMBIOS", "->", motivo)

In [ ]:
# Guarda la decisión en ml_monitoreo.
clave = f"{lote_actual}|{caso}"
fila = [(clave, lote_actual, caso, meta["version"] if meta else None, int(n_drift), metrica_base, metrica_nueva,
         bool(degradacion), bool(reentrenar), motivo)]
esquema_monitoreo = ("clave string, lote string, caso string, champion_version int, n_features_drift int, "
                     "metrica_base double, metrica_nueva double, degradacion boolean, reentrenar boolean, motivo string")
monitoreo = spark.createDataFrame(fila, esquema_monitoreo)

monitoreo.write.mode("overwrite").option("replaceWhere", f"caso = '{caso}' AND lote = '{lote_actual}'").saveAsTable("ml_monitoreo")

In [ ]:
# Termina aquí si no hace falta recalibrar.
if not reentrenar:
    dbutils.notebook.exit("Sin cambios: se conserva el champion")

## 9. Entrenamiento

In [ ]:
# Separa train y holdout; da más peso al lote actual.
holdout = df.filter(col("es_holdout"))
train = (df.filter(~col("es_holdout"))
         .withColumn("peso", when(col("lote") == lote_actual, lit(3.0)).otherwise(lit(1.0))))
snapshots_train = sorted(r[0] for r in train.select("lote").distinct().collect())

print("train:", train.count(), "| holdout:", holdout.count())
print("Lotes de entrenamiento:", snapshots_train)

In [ ]:
# Prepara las variables: vector y escalado.
num_assembler = VectorAssembler(inputCols=cols_nums, outputCol="num_features")
scaler = StandardScaler(inputCol="num_features", outputCol="features_scaled", withMean=True, withStd=True)

preparacion = [num_assembler, scaler]

In [ ]:
# Algoritmos candidatos, con los mismos datos y pesos.
comunes = dict(labelCol=target, featuresCol="features_scaled", weightCol="peso")

algoritmos = {
    "LogisticRegression": LogisticRegression(maxIter=100, **comunes),
    "DecisionTree": DecisionTreeClassifier(maxDepth=6, seed=SEED, **comunes),
    "RandomForest": RandomForestClassifier(numTrees=100, maxDepth=8, seed=SEED, **comunes),
    "GBT": GBTClassifier(maxIter=50, maxDepth=5, seed=SEED, **comunes),
}

In [ ]:
# Entrena cada algoritmo y predice sobre el holdout.
modelos, predicciones = {}, {}

for nombre, estimador in algoritmos.items():
    pipeline = Pipeline(stages=preparacion + [estimador])
    modelos[nombre] = pipeline.fit(train)
    predicciones[nombre] = modelos[nombre].transform(holdout)
    print("entrenado:", nombre)

if champion is not None:
    predicciones["champion"] = champion.transform(holdout)

## 10. Evaluación en el holdout

In [ ]:
# Compara los modelos en el holdout.
evaluador = MulticlassClassificationEvaluator(labelCol=target, predictionCol="prediction")

filas = []
for nombre, pred in predicciones.items():
    filas.append({
        "Modelo": nombre,
        "AUC": evaluador_auc.evaluate(pred),
        "Accuracy": evaluador.setMetricName("accuracy").evaluate(pred),
        "F1": evaluador.setMetricName("f1").evaluate(pred),
        "Precision_P": evaluador.setMetricName("precisionByLabel").setMetricLabel(1.0).evaluate(pred),
        "Recall_P": evaluador.setMetricName("recallByLabel").setMetricLabel(1.0).evaluate(pred),
    })

tabla = pd.DataFrame(filas).sort_values("AUC", ascending=False).reset_index(drop=True)
display(tabla.round(4))

## 11. Promoción champion / challenger

In [ ]:
# Promueve al challenger solo si mejora al champion más que el umbral.
challengers = tabla[tabla["Modelo"] != "champion"].reset_index(drop=True)
ganador = challengers.loc[0, "Modelo"]
auc_ganador = float(challengers.loc[0, "AUC"])

if champion is None:
    promover = True
else:
    auc_champion = float(tabla.loc[tabla["Modelo"] == "champion", "AUC"].iloc[0])
    promover = auc_ganador >= auc_champion + UMBRALES["mejora_minima_auc"]

print("Mejor algoritmo:", ganador, "| AUC:", round(auc_ganador, 4))
print("Promover challenger a champion:", promover)

## 12. Falsas alarmas del mejor algoritmo

In [ ]:
# Falsas alarmas por tipo de ventana y matriz de confusión.
pred_ganador = predicciones[ganador]
falsas = (pred_ganador.where(col(target) == 0).groupBy("tipo").agg((100 * F.avg("prediction")).alias("pct_falsas_alarmas"))
          .toPandas().set_index("tipo").pct_falsas_alarmas)
matriz = (pred_ganador.groupBy(target, "prediction").count().toPandas()
          .pivot(index=target, columns="prediction", values="count").fillna(0).astype(int))
print("Falsas alarmas (%) por tipo de ventana negativa:", falsas.round(2).to_dict())
display(matriz)

## 13. Guardar el modelo en el Volume y registrarlo

In [ ]:
# Métricas y detalle que se guardan con el modelo.
metricas = challengers.loc[0, ["AUC", "Accuracy", "F1", "Precision_P", "Recall_P"]].astype(float).to_dict()
detalle = {"comparacion": challengers.round(4).to_dict("records"), "falsas_alarmas_pct": falsas.round(3).to_dict()}

In [ ]:
# Guarda el modelo en el Volume y lo registra.
version = registro.count() + 1
ruta = f"{volumen}/modelos/{caso}/v{version}"
modelos[ganador].write().overwrite().save(ruta)

fila = [(caso, version, ganador, lote_actual, ruta, bool(promover), json.dumps(cols_nums), json.dumps([]),
         json.dumps(snapshots_train), json.dumps(metricas), json.dumps(detalle, default=float))]
esquema_registro = ("caso string, version int, algoritmo string, lote string, ruta string, promovido boolean, "
                    "columnas_num string, columnas_cat string, snapshots string, metricas string, detalle string")
spark.createDataFrame(fila, esquema_registro).write.mode("append").saveAsTable("ml_registro_modelos")

print(f"Version v{version} guardada en {ruta}")
print("Estado:", "CHAMPION" if promover else "challenger (se conserva el champion actual)")

## 14. MLflow — experimento
Cada ejecución del notebook queda registrada en un experimento de MLflow (un *run* padre por ejecución y un *run* hijo por algoritmo) para comparar parámetros y métricas desde la interfaz.

In [ ]:
# Experimento de MLflow del caso.
usuario = spark.sql("SELECT current_user()").first()[0]
mlflow.set_experiment(f"/Users/{usuario}/sismos_{caso}")

In [ ]:
# Registra parámetros, métricas y modelo en MLflow.
with mlflow.start_run(run_name=f"{caso}_{lote_actual}"):
    mlflow.log_params({"lote": lote_actual, "forzado": forzar, "n_train": train.count(), "n_holdout": holdout.count(),
                       "snapshots_train": ",".join(snapshots_train), "variables": ",".join(cols_nums),
                       "ganador": ganador, "promovido": bool(promover), "version_registro": version, "ruta_modelo": ruta})
    mlflow.log_metric("auc_ganador", auc_ganador)
    if champion is not None:
        mlflow.log_metric("auc_champion", auc_champion)
    mlflow.log_text(challengers.to_csv(index=False), "comparacion.csv")

    for _, fila_modelo in challengers.iterrows():
        with mlflow.start_run(run_name=fila_modelo["Modelo"], nested=True):
            mlflow.log_param("algoritmo", fila_modelo["Modelo"])
            mlflow.log_metrics({"auc": float(fila_modelo["AUC"]), "accuracy": float(fila_modelo["Accuracy"]),
                                "f1": float(fila_modelo["F1"]), "recall_p": float(fila_modelo["Recall_P"])})